# Sélection de Variables (Feature Importance)

Analyse de l'importance des features TF-IDF via Random Forest, sélection d'un sous-ensemble optimal, et comparaison avant/après.

**Checklist SAE-118 :**
- Feature importance (Random Forest)
- Visualisation top-N features
- Sélection de sous-ensembles (500, 1000, 2000)
- Comparaison performance avant/après

In [ ]:
import sys
sys.path.insert(0, '../..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.constants import RANDOM_STATE, SCORE_NAMES, POLARITY_NAMES
from src.ml_utils import load_and_prepare, split_data
from src.evaluation import plot_confusion, print_report
from src import setup_plot_style

setup_plot_style()
MODELS_DIR = '../../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

## 1. Chargement et Préparation

In [ ]:
df = load_and_prepare()
data = split_data(df['text'], df['polarity'], df['stars'])
print(f"Train: {len(data['X_train'])} | Val: {len(data['X_val'])} | Test: {len(data['X_test'])}")

## 2. Vectorisation TF-IDF (baseline 10k features)

In [ ]:
N_FEATURES_FULL = 10_000

vectorizer_full = TfidfVectorizer(
    max_features=N_FEATURES_FULL, min_df=5, max_df=0.7, ngram_range=(1, 2)
)

X_train_full = vectorizer_full.fit_transform(data['X_train'])
X_val_full = vectorizer_full.transform(data['X_val'])
X_test_full = vectorizer_full.transform(data['X_test'])

feature_names = vectorizer_full.get_feature_names_out()
print(f'Matrice TF-IDF (train) : {X_train_full.shape}')

## 3. Modèle Baseline (toutes les features)

In [ ]:
clf_baseline = LogisticRegression(max_iter=500, random_state=RANDOM_STATE, n_jobs=-1)
clf_baseline.fit(X_train_full, data['y_sc_train'])

y_pred_baseline = clf_baseline.predict(X_val_full)
acc_baseline = accuracy_score(data['y_sc_val'], y_pred_baseline)
f1_baseline = f1_score(data['y_sc_val'], y_pred_baseline, average='macro')

print(f'=== BASELINE ({N_FEATURES_FULL} features) ===')
print(f'Accuracy (val) : {acc_baseline:.4f}')
print(f'F1 Macro (val) : {f1_baseline:.4f}')

## 4. Feature Importance (Random Forest)

In [ ]:
print('Entraînement Random Forest pour feature importance...')
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_full, data['y_sc_train'])

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
print(f'Feature importance calculée sur {len(importances)} features.')

## 5. Visualisation des Top-N Features

In [ ]:
TOP_N = 30
top_features = [feature_names[i] for i in indices[:TOP_N]]
top_scores = importances[indices[:TOP_N]]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(TOP_N), top_scores[::-1], color='steelblue', edgecolor='white')
ax.set_yticks(range(TOP_N))
ax.set_yticklabels(top_features[::-1], fontsize=10)
ax.set_xlabel('Importance (Random Forest)')
ax.set_title(f'Top {TOP_N} Features les Plus Importantes', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution cumulée
sorted_imp = np.sort(importances)[::-1]
cumulative = np.cumsum(sorted_imp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sorted_imp[:500], color='steelblue')
axes[0].set_xlabel('Rang de la feature')
axes[0].set_ylabel('Importance')
axes[0].set_title("Distribution de l'importance (top 500)")
axes[0].grid(alpha=0.3)

axes[1].plot(cumulative, color='darkorange')
for t in [0.5, 0.7, 0.9, 0.95]:
    n = np.searchsorted(cumulative, t) + 1
    axes[1].axhline(t, color='gray', linestyle='--', linewidth=0.8)
    axes[1].annotate(f'{t*100:.0f}% -> {n} feat.', xy=(n, t),
                     xytext=(n + 200, t - 0.03), fontsize=8,
                     arrowprops=dict(arrowstyle='->', color='gray'))
axes[1].set_xlabel('Nombre de features')
axes[1].set_ylabel('Importance cumulée')
axes[1].set_title('Importance Cumulée')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Sélection de Sous-ensembles & Comparaison

In [ ]:
K_VALUES = [500, 1000, 2000]
results = {}

for k in K_VALUES:
    top_k_idx = indices[:k]
    X_tr_k = X_train_full[:, top_k_idx]
    X_vl_k = X_val_full[:, top_k_idx]

    clf_k = LogisticRegression(max_iter=500, random_state=RANDOM_STATE, n_jobs=-1)
    clf_k.fit(X_tr_k, data['y_sc_train'])

    y_pred_k = clf_k.predict(X_vl_k)
    acc_k = accuracy_score(data['y_sc_val'], y_pred_k)
    f1_k = f1_score(data['y_sc_val'], y_pred_k, average='macro')

    results[f'Top-{k}'] = {
        'n_features': k, 'accuracy': acc_k, 'f1_macro': f1_k,
        'clf': clf_k, 'top_k_idx': top_k_idx
    }
    print(f'Top-{k:5d} features | Acc: {acc_k:.4f} | F1 Macro: {f1_k:.4f}')

results['Baseline (10k)'] = {
    'n_features': N_FEATURES_FULL, 'accuracy': acc_baseline, 'f1_macro': f1_baseline
}

In [ ]:
# Tableau comparatif
summary = pd.DataFrame([
    {'Config': name, 'Features': v['n_features'],
     'Accuracy': round(v['accuracy'], 4), 'F1 Macro': round(v['f1_macro'], 4),
     'Réduction (%)': round((1 - v['n_features'] / N_FEATURES_FULL) * 100, 1)}
    for name, v in results.items()
])
display(summary.sort_values('F1 Macro', ascending=False).reset_index(drop=True))

# Visualisation
configs = list(results.keys())
f1s = [results[c]['f1_macro'] for c in configs]
accs = [results[c]['accuracy'] for c in configs]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(configs))
ax.bar(x - 0.15, accs, 0.3, label='Accuracy', color='steelblue', alpha=0.85)
ax.bar(x + 0.15, f1s, 0.3, label='F1 Macro', color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(configs, rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Performance par Nombre de Features')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Meilleur Sous-ensemble & Test Final

In [ ]:
# Meilleure config réduite
reduced = {k: v for k, v in results.items() if k != 'Baseline (10k)'}
best_name = max(reduced, key=lambda k: reduced[k]['f1_macro'])
best = reduced[best_name]

print(f'=== MEILLEURE CONFIG : {best_name} ===')
print(f'Features   : {best["n_features"]}')
print(f'Accuracy   : {best["accuracy"]:.4f}')
print(f'F1 Macro   : {best["f1_macro"]:.4f}')
print(f'Réduction  : {(1 - best["n_features"] / N_FEATURES_FULL) * 100:.1f}%')
print(f'\nΔ F1 vs baseline : {best["f1_macro"] - f1_baseline:+.4f}')

# Test final
X_test_k = X_test_full[:, best['top_k_idx']]
print_report(data['y_sc_test'], best['clf'].predict(X_test_k),
             SCORE_NAMES, f'{best_name} — Score (Test)')
plot_confusion(data['y_sc_test'], best['clf'].predict(X_test_k),
               SCORE_NAMES, f'{best_name} — Score (Test)')

## 8. Sauvegarde

In [ ]:
selected_names = feature_names[best['top_k_idx']]

joblib.dump(best['clf'], os.path.join(MODELS_DIR, 'best_selected_classifier.pkl'))
joblib.dump(selected_names, os.path.join(MODELS_DIR, 'selected_feature_names.pkl'))
joblib.dump(vectorizer_full, os.path.join(MODELS_DIR, 'tfidf_vectorizer_full.pkl'))

print(f'Modèle sauvegardé         : {MODELS_DIR}best_selected_classifier.pkl')
print(f'Features sélectionnées    : {MODELS_DIR}selected_feature_names.pkl ({len(selected_names)})')
print(f'Vectorizer sauvegardé     : {MODELS_DIR}tfidf_vectorizer_full.pkl')